In [1]:
from langgraph.graph import StateGraph,END
import random
from typing import Dict,List,TypedDict

In [2]:
class AgentState(TypedDict):
    name:str
    number:List[int]
    counter:int

In [ ]:
def greeting_node(state:AgentState)->AgentState:
    """Greeting node which says hi to a person"""
    state["name"]=f"hi there,{state["name"]}"
    state["counter"]=0
    return state

def random_node(state: AgentState) -> AgentState:
    """Generates a random number from 0 to 10"""
    state["number"].append(random.randint(0, 10))
    state["counter"] += 1

    return state

def should_continue(state: AgentState) -> AgentState:
    """Function to decide what to do next"""
    if state["counter"] < 5: #5 times the random node will interate
        print("ENTERING LOOP", state["counter"])
        return "loop"  # Continue looping
    else:
        return "exit"  # Exit the loop

In [ ]:
# greeting → random → random → random → random → random → END

In [4]:
graph = StateGraph(AgentState)

graph.add_node("greeting", greeting_node)
graph.add_node("random", random_node)
graph.add_edge("greeting", "random")


graph.add_conditional_edges(
    "random",     # Source node
    should_continue, # Action
    {
        "loop": "random",  
        "exit": END          
    }
)

graph.set_entry_point("greeting")

app = graph.compile()

In [5]:
#Visualizing this agent code
from IPython.display import Markdown, display
display(Markdown(f"```mermaid\n{app.get_graph().draw_mermaid()}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	greeting(greeting)
	random(random)
	__end__([<p>__end__</p>]):::last
	__start__ --> greeting;
	greeting --> random;
	random -. &nbsp;exit&nbsp; .-> __end__;
	random -. &nbsp;loop&nbsp; .-> random;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
app.invoke({"name":"Vaibhav", "number":[], "counter":-100})

ENTERING LOOP 1
ENTERING LOOP 2
ENTERING LOOP 3
ENTERING LOOP 4


{'name': 'hi there,Vaibhav', 'number': [3, 0, 10, 4, 5], 'counter': 5}